# Carvana EDA And Results

In [1]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
PACKAGE_ROOT = PROJECT_ROOT.parent

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from carvana.core.config import CarvanaConfig
from carvana.datasets.carvana_data import build_splits, load_mask, load_rgb
from carvana.visualization.plots import plot_histories

/Users/lunis/PycharmProjects/machine_learning/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:989: The handshake operation timed out>
  data = fetch_version_info()


In [ ]:
config = CarvanaConfig()
config.debug_max_samples = None

df, train_df, val_df = build_splits(config)

print("workdir:", config.workdir)
print("total:", len(df))
print("train:", len(train_df), "images |", train_df["car_id"].nunique(), "cars")
print("val:", len(val_df), "images |", val_df["car_id"].nunique(), "cars")
print("car_id intersection:", len(set(train_df["car_id"]) & set(val_df["car_id"])))

In [ ]:
display(df.head())
display(df[["car_id", "angle_id"]].describe(include="all").T)

In [ ]:
sample_mask_paths = df["mask_path"].sample(min(256, len(df)), random_state=config.seed)
mask_coverages = []

for path in sample_mask_paths:
    mask = (load_mask(path) > 127).astype(np.float32)
    mask_coverages.append(mask.mean())

mask_coverages = np.array(mask_coverages)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(df["car_id"].value_counts(), bins=16)
axes[0].set_title("Images per car")
axes[1].hist(df["angle_id"].dropna(), bins=16)
axes[1].set_title("Angle distribution")
axes[2].hist(mask_coverages, bins=20)
axes[2].set_title("Foreground coverage")
plt.tight_layout()
plt.show()

print("mean mask coverage:", float(mask_coverages.mean()))
print("median mask coverage:", float(np.median(mask_coverages)))

In [ ]:
def show_random_samples(frame: pd.DataFrame, n: int = 4, seed: int = 42) -> None:
    sample = frame.sample(min(n, len(frame)), random_state=seed).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 3, figsize=(14, 4 * len(sample)))

    if len(sample) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, row in sample.iterrows():
        image = load_rgb(row["image_path"])
        mask = (load_mask(row["mask_path"]) > 127).astype(np.uint8)
        overlay = image.copy()
        overlay[mask == 1] = (0.5 * overlay[mask == 1] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)

        axes[i, 0].imshow(image)
        axes[i, 0].set_title(f"Image\n{row['id']}")
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title("Mask")
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay")

        for j in range(3):
            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()


show_random_samples(train_df, n=3, seed=config.seed)
show_random_samples(val_df, n=3, seed=config.seed + 1)

## Training Curves

In [ ]:
report_dir = config.report_dir
history_paths = {
    "BCE": report_dir / "smp_unet_bce_history.csv",
    "DiceLoss": report_dir / "smp_unet_dice_history.csv",
}

histories = {}
for name, path in history_paths.items():
    if path.exists():
        histories[name] = pd.read_csv(path)
        print("loaded:", path)
    else:
        print("missing:", path)

list(histories)

In [ ]:
if histories:
    plot_histories(histories, "train_loss")
    plot_histories(histories, "val_loss")
    plot_histories(histories, "val_iou")
    plot_histories(histories, "val_dice")
else:
    print("History CSV files not found yet. Run training first.")

## Final Metrics

In [ ]:
summary_path = report_dir / "unet_summary.csv"

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display(summary_df.sort_values("val_dice_mean", ascending=False))
else:
    print("Summary file not found:", summary_path)